<a href="https://colab.research.google.com/github/kritikasharma-ks/BuildFastWithAI/blob/main/Execerise_ChatGPT_Replica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Exercise 1 : Display Recent Conversation
Modify the `display_chat_history` function to allow users to specify a range of messages to display, such as showing only the last 10 messages.  



####Hint:


In [ ]:
!pip install -qU "langchain>=0.3.6" "langchain-core>=0.3.15" "langchain-google-genai>=2.0.1" "langchain-classic>=0.3.6"


In [22]:
from langchain_classic.memory import ChatMessageHistory
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
gemini_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=GOOGLE_API_KEY)

In [19]:
# 1. Prompt Template
prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name='chat_history'),
    'human', '{input}'
])

# 2. Chain with LLM Model and Prompt

chain = prompt | gemini_model

# 3. Store chat history

history_store = {}

def get_session_history(session_id: str):
  if session_id not in history_store:
    history_store[session_id] = ChatMessageHistory()
  return history_store[session_id]

# 4. Wrap with RunnableWithMessageHistory

conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key='input',
    history_messages_key='chat_history'
)

# 5. Example usage

session_id = 'default'
response = conversation.invoke(
    {'input': 'Hello'},
    config={'configurable': {'session_id':session_id}}
)

# 6. Last 10 conversations in chat history

n = 10

history = get_session_history(session_id)
chat_history = history.messages

chat_history_new = chat_history[-n:]

# 7. Print conversations

for msg in chat_history_new:
  print(f'{msg.type.upper()}:{msg.content}')

HUMAN:Hello
AI:Hello! How can I help you today?


### Exercise 2 : Export Chat History
Add a feature that allows users to export the chat history to a file in a structured format, such as JSON or CSV.  



####Hint:

In [21]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import json
import csv

In [33]:
# call llm and save required chat_history
structured_data = [
        {"role": "Human" if isinstance(msg, HumanMessage) else "AI",
         "content": msg.content}
        for msg in chat_history
    ]

# now save is json/csv file

with open('chat_history.json', 'w') as f:
    json.dump(structured_data, f, indent=4)

with open('chat_history.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Role', 'Content'])
    for msg in structured_data:
        writer.writerow([msg['role'], msg['content']])

for msg in structured_data:
  print(f'{msg['role'].upper()}: {msg['content']}')

HUMAN: Hello
AI: Hello! How can I help you today?
